<a href="https://colab.research.google.com/github/isaacmutuma/cifar-10/blob/main/cifar10_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#Imports
import torch
from torch import nn
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

#we define what our model will have layers and how the data will flow(forward)
class SimpleCNN(nn.Module):
  def __init__ (self):
    super().__init__()
    self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),# 3 colors coming in
            #0ut goes 32 diffrent patterns , the area to be scan at time is 3 by 3 a kernel
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
         )
    self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
#how data will flow through the layers
  def forward(self, x):
       x = self.conv_block1(x)
       x = self.conv_block2(x)
       x = self.classifier(x)
       return x

# now lets load this dat we have already made way for
training_data= datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)
test_data= datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)
print(len(training_data))
print(training_data[0][0].shape)
print(training_data[0][1])

training_loader=DataLoader(training_data, batch_size=64,shuffle=True)
test_loader=DataLoader(test_data, batch_size=64,shuffle=False)

# where will our model run n what device and then bring it to life
device="cuda" if torch.cuda.is_available() else "cpu"
model = SimpleCNN().to(device)

#loss function and optimizer
loss_fn= nn.CrossEntropyLoss() # single value when pred is compared to the actual value
optimizer= torch.optim.Adam(model.parameters(), lr=1e-3) #params are the weights of the pixels+ grad from loss.backward()
                                                        # the output is the weights readjusted to fit the grad

#How exactly do we find the correct weights for every pixel we train
def train(training_loader,model,loss_fn,optimizer):
  model.train() # in training mode now
  # we need to get the data as a tensor(X) and the label y
  for batch,(X,y) in enumerate(training_loader):
    X,y = X.to(device),y.to(device)# move the batch to the device
    # the forward pass
    pred= model(X)
    #loss calculation pass in the prediction and the correct labels
    loss=loss_fn(pred,y) # we have a single number to measure wrongness

    #backward pass
    optimizer.zero_grad() #wipes the gradients from last batch
    loss.backward() # gradients are calculated
    optimizer.step()# reads the weights and gradients and makes adjustements

    if batch % 100 == 0: #prints about the 10 losses because the epoch about 900
      print(f"loss: {loss.item()} [{batch * len(X)}/{len(training_loader.dataset)}]")
      # the actual loss and the number of images processed over the total number in th dataset

def test(test_loader, model ):
  model.eval()# in evaluation mode
  correct=0

  with torch.no_grad():# we dont need to keep a trail of the forward pass
    for X,y in test_loader:
       X, y = X.to(device), y.to(device)
       pred = model(X) #the shape is (64,10) #for all 64 images each has a score for everyclass
       correct += (pred.argmax(1) == y).type(torch.float).sum().item()
                  #highest scores position in scores dimension and compares it the correct label
                  #converted to float and sum the floats up and the number of 1s are the correct predictions

  correct /= len(test_loader.dataset) # take the correct values and divide them by the datasets total
  print(f"Accuracy: {(100*correct):>0.1f}%") # multiply by 100 to get the percentage then to 1dp
# the loop to run all batches
epochs = 10 # number of times you learn and test
for t in range(epochs):
     print(f"Epoch {t+1}")
     train(training_loader, model, loss_fn, optimizer)
     test(test_loader, model)
print("Done!")















50000
torch.Size([3, 32, 32])
6
Epoch 1
loss: 2.308537483215332 [0/50000]
loss: 1.639224648475647 [6400/50000]
loss: 1.466172456741333 [12800/50000]
loss: 1.5330904722213745 [19200/50000]
loss: 1.5780109167099 [25600/50000]
loss: 1.3091516494750977 [32000/50000]
loss: 1.2379209995269775 [38400/50000]
loss: 1.16990327835083 [44800/50000]
Accuracy: 59.4%
Epoch 2
loss: 0.9764253497123718 [0/50000]
loss: 0.9921190142631531 [6400/50000]
loss: 1.0310524702072144 [12800/50000]
loss: 1.1430243253707886 [19200/50000]
loss: 1.1283104419708252 [25600/50000]
loss: 1.0227049589157104 [32000/50000]
loss: 1.0044552087783813 [38400/50000]
loss: 1.0389516353607178 [44800/50000]
Accuracy: 65.9%
Epoch 3
loss: 0.9879915714263916 [0/50000]
loss: 0.8237802982330322 [6400/50000]
loss: 0.9712952375411987 [12800/50000]
loss: 0.9562773704528809 [19200/50000]
loss: 0.9176832437515259 [25600/50000]
loss: 0.913940966129303 [32000/50000]
loss: 0.7113152146339417 [38400/50000]
loss: 0.8688564896583557 [44800/50000]
